# 04. Regresión logística y desempeño predictivo

Este notebook ejecuta el modelo multivariado actualizado. El modelo principal es preespecificado por plausibilidad clínica y epidemiológica; la eliminación hacia atrás se conserva únicamente como análisis de sensibilidad. También se evalúan colinealidad, linealidad del logit, calibración y discriminación con validación interna por bootstrap.

In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'src'))

from src.export_tables_pdf import (
    build_calibration_tables,
    build_model_outputs,
    export_pdf,
    load_clean_data,
)

pd = __import__('pandas')
pd.set_option('display.max_columns', None)

In [2]:
df_clean = load_clean_data()
outputs = build_model_outputs(df_clean)
calibration_table, calibration_summary = build_calibration_tables(outputs['y'], outputs['predictions'])
print(f'Observaciones incluidas en el modelo: {len(outputs["y"]):,}')

Observaciones incluidas en el modelo: 405


## Modelo logístico principal

In [3]:
outputs['regression_table']

,Variable,Coeficiente (B),Error Estándar,Valor p,OR,IC 95% Inferior,IC 95% Superior
0,Intercepto,0.745,2.034,0.714,2.11,0.04,113.35
1,Edad,0.147,0.028,< 0.001,1.16,1.10,1.23
2,IMC,-0.326,0.047,< 0.001,0.72,0.66,0.79
3,Sexo: mujer vs hombre,1.388,0.459,0.002,4.01,1.63,9.84
4,Trabaja: sí vs no,0.259,0.412,0.529,1.30,0.58,2.90
5,Enfermedad: sí vs sano,0.668,0.445,0.133,1.95,0.82,4.67
6,Actividad física: sí vs no,0.092,0.352,0.794,1.10,0.55,2.18


## Diagnósticos del modelo

In [4]:
outputs['vif_table']

,Variable,VIF
0,edad,1.25
1,imc,1.10
2,sexo,1.08
3,trabaja,1.23
4,enfermedad,1.10
5,realiza_af,1.06


In [5]:
outputs['linearity_table']

,Variable,Término,Valor p,Interpretación
0,edad,edad_log,0.9569,Sin evidencia de no linealidad
1,imc,imc_log,0.8938,Sin evidencia de no linealidad


## Calibración

In [6]:
calibration_summary

,Métrica,Valor
0,Brier score,0.084
1,Intercepto de calibración,-0.000
2,Pendiente de calibración,1.000


In [7]:
calibration_table

,Decil,n,Probabilidad predicha media,Proporción observada
0,1,41,0.330,0.317
1,2,40,0.627,0.750
2,3,41,0.799,0.683
3,4,40,0.893,0.875
4,5,41,0.930,0.927
5,6,40,0.963,1.000
6,7,40,0.977,0.950
7,8,41,0.988,1.000
8,9,40,0.994,1.000
9,10,41,0.998,1.000


## Discriminación y validación interna

In [8]:
outputs['auc_table']

,Métrica,Valor
0,AUC aparente,0.892
1,IC 95% bootstrap del AUC aparente,0.850 - 0.928
2,Optimismo promedio,0.013
3,AUC corregida por optimismo,0.879
4,Remuestreos exitosos,200 / 200


## Análisis de sensibilidad: eliminación hacia atrás

In [9]:
outputs['backward_table']

,Iteración,Variables,Variable con mayor p,p máximo,Acción
0,1,"edad, imc, sexo, trabaja, enfermedad, realiza_af",realiza_af,0.7944,Eliminar
1,2,"edad, imc, sexo, trabaja, enfermedad",trabaja,0.5339,Eliminar
2,3,"edad, imc, sexo, enfermedad",enfermedad,0.1528,Eliminar
3,4,"edad, imc, sexo",sexo,0.0024,Detener


## Regeneración de artefactos

In [11]:
output_pdf = export_pdf()
print(f'PDF, CSV, curva ROC, calibración e informe regenerados desde: {output_pdf}')

PDF, CSV, curva ROC, calibración e informe regenerados desde: C:\Users\marco\Documents\analysis_osteoporosis\results\tablas_resultados_apa.pdf
